# 01 - Introducao ao MCP
> O que e o Model Context Protocol e por que importa

**Modulo:** `05_mcp` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

## O que e MCP?
Padrao aberto da Anthropic para conectar LLMs a ferramentas de forma padronizada.

Sem MCP: Claude <-> Tool A customizada <-> Tool B customizada
Com MCP: Claude <-> MCP Client <-> MCP Server A <-> Servico A


In [ ]:
import os, json, tempfile

WS=tempfile.mkdtemp()

class MCPServer:
    def __init__(self,nome): self.nome=nome; self.tools={}
    def tool(self,nome,fn,desc,schema): self.tools[nome]={'fn':fn,'desc':desc,'schema':schema}
    def list_tools(self): return [{'name':n,'description':t['desc'],'input_schema':t['schema']} for n,t in self.tools.items()]
    def call(self,nome,args): return self.tools[nome]['fn'](**args) if nome in self.tools else 'Tool nao encontrada'

srv=MCPServer('filesystem')
srv.tool('write',
    lambda path,content: open(os.path.join(WS,path),'w',encoding='utf-8').write(content) or 'OK',
    'Escreve arquivo.',
    {'type':'object','properties':{'path':{'type':'string'},'content':{'type':'string'}},'required':['path','content']})
srv.tool('read',
    lambda path: open(os.path.join(WS,path),encoding='utf-8').read() if os.path.exists(os.path.join(WS,path)) else 'Nao encontrado.',
    'Le arquivo.',
    {'type':'object','properties':{'path':{'type':'string'}},'required':['path']})

tools_api=srv.list_tools()

def agente(q):
    msgs=[{'role':'user','content':q}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=tools_api,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                res.append({'type':'tool_result','tool_use_id':b.id,'content':str(srv.call(b.name,b.input))})
        msgs.append({'role':'user','content':res})

print(agente('Crie um arquivo notas.txt com: Estudar MCP hoje.'))
print(agente('Leia o arquivo notas.txt.'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
